# 03 — Score Trends

Seasonal growth curves and ensemble score trajectories.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('../scores.db')
df = pd.read_sql('SELECT * FROM performances', conn)
df['performance_date'] = pd.to_datetime(df['performance_date'])
df['year'] = df['performance_date'].dt.year
df['week'] = df['performance_date'].dt.isocalendar().week
print(df.shape)

In [ ]:
# Median total score by year and class
yearly = df.groupby(['year', 'class_code'])['total_score'].median().unstack()
yearly.plot(figsize=(12, 5), marker='o', title='Median total score by year (per class)')
plt.tight_layout()

In [ ]:
# Within-season score growth: score vs week-of-year, averaged across ensembles
weekly = df.groupby(['year', 'week'])['total_score'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for yr, grp in weekly.groupby('year'):
    ax.plot(grp['week'], grp['total_score'], label=yr, alpha=0.7)
ax.set_xlabel('Week of year')
ax.set_ylabel('Mean total score')
ax.set_title('Within-season score growth by year')
ax.legend()
plt.tight_layout()

In [ ]:
# Top ensembles by number of seasons competed
seasons_per_ensemble = df.groupby('ensemble_slug')['year'].nunique().sort_values(ascending=False)
top_ensembles = seasons_per_ensemble.head(10).index.tolist()
print(top_ensembles)

In [ ]:
# Score trajectory for top ensembles (finals / championship performances only)
finals = df[df['ensemble_slug'].isin(top_ensembles)].copy()
# Keep one score per ensemble per year (highest total — typically the finals performance)
peak = finals.groupby(['ensemble_slug', 'year'])['total_score'].max().reset_index()

fig, ax = plt.subplots(figsize=(13, 6))
for slug, grp in peak.groupby('ensemble_slug'):
    ax.plot(grp['year'], grp['total_score'], marker='o', label=slug)
ax.set_title('Peak annual score — top 10 ensembles by seasons competed')
ax.set_xlabel('Year')
ax.set_ylabel('Peak total score')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()